# Tăng cường đặc trưng VSL

Tạo bản nhiễu từ mẫu gốc trong `data/processed`; `data/raw` giữ nguyên. Đặt `MIN_SAMPLES_PER_CLASS = N` để chỉ bổ sung các lớp có ít hơn N mẫu hoàn chỉnh trong `processed`, tạo đúng số còn thiếu. Đặt `-1` để tăng cường tất cả mẫu gốc theo `COPIES_PER_SAMPLE` mà không giới hạn. Mỗi bản mới ghi `origin_sample_id` để đi cùng fold với mẫu gốc. Validation trong `train.ipynb` chỉ chấm mẫu gốc.


In [1]:
import importlib, sys
from pathlib import Path
import numpy as np
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src').is_dir()), None)
if PROJECT_ROOT is None: raise FileNotFoundError('Chạy Jupyter từ dự án hoặc notebooks/')
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))
reader = importlib.reload(importlib.import_module('src.data.data_reader'))
augment = importlib.reload(importlib.import_module('src.data.augment'))
DataHandler = reader.DataHandler
PROCESSED_ROOT = PROJECT_ROOT / 'data' / 'processed'
RAW_ROOT = PROJECT_ROOT / 'data' / 'raw'
SELECTED_CLASSES = 'all'  # 'all', 'xin chào', hoặc ['xin chào', 'cảm ơn']
MIN_SAMPLES_PER_CLASS = int(input("MIN_SAMPLES_PER_CLASS (-1 to all): "))  # N; -1 = tăng cường toàn bộ, không giới hạn
COPIES_PER_SAMPLE = 2    # chỉ dùng khi MIN_SAMPLES_PER_CLASS = -1
NOISE_STRENGTH = 1.0    # khoảng (0, 3]
SEED = None  # đặt số nguyên để chạy tái lập
if MIN_SAMPLES_PER_CLASS < -1 or COPIES_PER_SAMPLE < 1 or not 0 < NOISE_STRENGTH <= 3: raise ValueError('Cấu hình ngưỡng/số bản/noise không hợp lệ')
classes = None if SELECTED_CLASSES == 'all' else ([SELECTED_CLASSES] if isinstance(SELECTED_CLASSES, str) else SELECTED_CLASSES)
plan, counts = augment.plan_augmentation(PROCESSED_ROOT, classes, MIN_SAMPLES_PER_CLASS, COPIES_PER_SAMPLE)
print(f'{sum(copies for _, copies in plan)} mẫu dự kiến tạo trong {len(counts)} lớp')


0 mẫu dự kiến tạo trong 14 lớp


## Kiểm tra trước khi tạo


In [2]:
for name, count in counts.items():
    extra = sum(copies for source, copies in plan if source.parent.name == name)
    print(f'{name}: hiện có {count}, thêm {extra}' + (f', sau đó có {count + extra}' if MIN_SAMPLES_PER_CLASS != -1 else ''))
print('Đã đủ ngưỡng, không cần tạo thêm.' if not plan else 'Raw và mẫu gốc được giữ nguyên. Chạy ô tiếp theo để tạo bản nhiễu.')


age: hiện có 45, thêm 0, sau đó có 45
cảm ơn: hiện có 45, thêm 0, sau đó có 45
g: hiện có 45, thêm 0, sau đó có 45
h: hiện có 45, thêm 0, sau đó có 45
k: hiện có 45, thêm 0, sau đó có 45
m: hiện có 45, thêm 0, sau đó có 45
n: hiện có 45, thêm 0, sau đó có 45
r: hiện có 45, thêm 0, sau đó có 45
t: hiện có 45, thêm 0, sau đó có 45
ten_la: hiện có 45, thêm 0, sau đó có 45
toi: hiện có 45, thêm 0, sau đó có 45
u: hiện có 45, thêm 0, sau đó có 45
xin chào: hiện có 45, thêm 0, sau đó có 45
địa chỉ: hiện có 45, thêm 0, sau đó có 45
Đã đủ ngưỡng, không cần tạo thêm.


## Tạo mẫu tăng cường
Ô này tính lại số mẫu hiện có ngay trước khi ghi. Với ngưỡng N, chạy lại sau khi đã đủ N sẽ không tạo thêm; với N = -1, mỗi lần chạy tạo thêm bản mới.


In [3]:
rng = np.random.default_rng(SEED)
run_plan, current_counts = augment.plan_augmentation(PROCESSED_ROOT, classes, MIN_SAMPLES_PER_CLASS, COPIES_PER_SAMPLE)
created, errors = [], []
for index, (source, copies) in enumerate(run_plan, 1):
    try:
        paths = augment.augment_sample(source, RAW_ROOT, copies, NOISE_STRENGTH, rng)
        created.extend(paths)
        print(f'[{index}/{len(run_plan)}] {source.parent.name}/{source.name} -> {", ".join(path.name for path in paths)}', flush=True)
    except Exception as exc:
        errors.append((str(source), str(exc)))
        print(f'[{index}/{len(run_plan)}] LỖI {source}: {exc}', flush=True)
print(f'Đã tạo {len(created)} mẫu; lỗi {len(errors)}')


Đã tạo 0 mẫu; lỗi 0


## Kiểm tra group


In [4]:
handler = DataHandler()
print('Mẫu đọc được:', handler.validate_dataset())
print('Tổng:', handler.get_class_counts())
print('Group gốc độc lập:', handler.get_origin_counts())
groups = np.asarray(handler.sample_groups('sample'))
augmented = np.asarray([handler.describe(i)['augmented'] for i in range(len(handler))])
assert all((~augmented[groups == group]).any() for group in np.unique(groups))
print(f'{augmented.sum()} bản tăng cường; mọi group đều có mẫu gốc')
print('Chạy lại train.ipynb từ ô đầu để dùng mẫu mới.')


Mẫu đọc được: 630
Tổng: {'age': 45, 'cảm ơn': 45, 'g': 45, 'h': 45, 'k': 45, 'm': 45, 'n': 45, 'r': 45, 't': 45, 'ten_la': 45, 'toi': 45, 'u': 45, 'xin chào': 45, 'địa chỉ': 45}
Group gốc độc lập: {'age': 15, 'cảm ơn': 15, 'g': 15, 'h': 15, 'k': 15, 'm': 15, 'n': 15, 'r': 15, 't': 15, 'ten_la': 15, 'toi': 15, 'u': 15, 'xin chào': 15, 'địa chỉ': 15}
420 bản tăng cường; mọi group đều có mẫu gốc
Chạy lại train.ipynb từ ô đầu để dùng mẫu mới.
